---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 2.1: Adding the First Tool


In the previous notebook we converted our recipe assistant from a raw completions call to an `Agent` object. The agent works, traces are captured, and eval still passes.

But it's still just a chatbot — it can only answer from its training data. Time for the first product conversation that changes that.

## 💬 Product Check-in: Current MLflow Version

> **Sam (Product):** "Small thing, but users keep asking what the current version of MLflow is and the agent either says it doesn't know or confidently states an old version number. It's making us look out of date."
>
> **You:** "That's a knowledge cutoff issue — the LLM's training data has a fixed snapshot of what 'current' means. It can't know the latest release without being told."
>
> **Sam:** "Can we just... tell it?"
>
> **You:** "Best way to do that reliably is a tool call. We build a `get_mlflow_version` function that returns the current version, and the agent calls it whenever that question comes up. Deterministic, always right, easy to update."
>
> **Sam:** "How do we know it's actually calling the tool?"
>
> **You:** "MLflow has scorers for exactly that — `ToolCallCorrectness` checks if the agent called the right tool with the right arguments, and `ToolCallEfficiency` catches redundant calls. I'll add both to our eval pipeline."
>
> **Sam:** "Ship it."

---

Let's do exactly that. We'll:
1. Build a `get_mlflow_version` tool
2. Add it to the agent
3. Add `ToolCallCorrectness` and `ToolCallEfficiency` to the eval pipeline
4. Write eval cases with `expected_tool_calls` ground truth

---
## 0 — Environment Setup

In [ ]:
import os
from dotenv import load_dotenv
import mlflow

load_dotenv()

MODEL = "gemini-2.5-flash-lite"
JUDGE_MODEL = "gemini:/gemini-3.1-flash-lite-preview"
EXPERIMENT_NAME = os.environ.get("EXPERIMENT_2_NAME", "mlflow-agent-2")

if not os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.langchain.autolog()


prompt_version = mlflow.genai.load_prompt("prompts:/mlflow-agent-system/2")
SYSTEM_PROMPT = prompt_version.template

# Convert completions call to langchain agent

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate
import mlflow

mlflow.langchain.autolog()

llm = ChatGoogleGenerativeAI(
    model=MODEL,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_agent(model=llm, tools=[], system_prompt=prompt)

def predict_fn(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content[0]["text"]

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "How many grams is 1 cup of granulated sugar?" }]})

In [ ]:
result["messages"][-1].content[0]["text"]

In [ ]:
version_eval_dataset = [
    {
        "inputs": {"question": "What is the current version of MLflow?"},
        "expectations": {
            "expected_facts": ["3.11.1"],
            "guidelines": [
                "Response must state a specific version number greater than 3",
            ],
        },
    },
    {
        "inputs": {"question": "Is MLflow 3.0 out yet?"},
        "expectations": {
            "expected_facts": ["yes"],
            "guidelines": [
                "Response must confirm MLflow 3.0 is available'",
            ],
        },
    },
    {
        "inputs": {"question": "What version of MLflow should I install for the genai features?"},
        "expectations": {
            "expected_facts": ["3.0"],
            "guidelines": [
                "Response must recommend a specific version number greater than 3",
                "Response must not suggest an outdated version like 2.x",
            ],
        },
    },
]

In [ ]:
from mlflow.genai.scorers import Correctness

results_v1 = mlflow.genai.evaluate(
    data=version_eval_dataset,
    predict_fn=predict_fn,
    scorers=[Correctness(model=f"gemini:/{JUDGE_MODEL}")],
)

results_v1

---
## 2 — Building the Version Tool

We'll build a tool that retrieves the latest version from PyPI.

The tool is deterministic and always correct — no hallucination possible. The LLM's job is just to figure out *when* to call it and *what arguments* to pass.

In [ ]:
import requests

def get_mlflow_version_pypi() -> str:
    """
    Fetches the current stable release of MLflow directly from PyPI.
    No API key required.
    """
    try:
        response = requests.get("https://pypi.org/pypi/mlflow/json", timeout=5)
        response.raise_for_status()
        return response.json()["info"]["version"]
    except Exception as e:
        return f"Could not fetch version from PyPI: {str(e)}"

In [ ]:
get_mlflow_version_pypi()

---
## 3 — Adding the Tool to the Agent

With the Agents SDK, adding a tool is one line — pass it in the `tools` list. The SDK reads the `@function_tool` docstring and generates the JSON schema for the LLM automatically.

We also update the system prompt to tell the agent about the tool. This matters — the LLM needs to know *when* to use it.

In [ ]:
tools = [get_mlflow_version_pypi]  # add tools here when ready

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

def agent_v2(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content[0]["text"]

In [ ]:
from mlflow.genai.scorers import Correctness

results_v1 = mlflow.genai.evaluate(
    data=version_eval_dataset,
    predict_fn=agent_v2,
    scorers=[Correctness(model=f"gemini:/{JUDGE_MODEL}")],
)

results_v1

In [ ]:
SYSTEM_PROMPT_WITH_TOOL = """You are a helpful cooking assistant for Mise en Place, a cooking app.
Answer questions about recipes, cooking techniques, ingredients, and meal planning.

You have access to a tool:
- convert_units: Convert between cooking measurement units (cups, grams, tbsp, etc.).
  Use this whenever the user asks about unit conversions. This tool is especially
  important for volume-to-weight conversions (like cups to grams) because these
  depend on the ingredient's density. Always use the tool instead of guessing.
"""

# Register prompt

In [ ]:
#run agent with new prompt and tool, evaluate again

### Making tools visible to MLflow

The key decorator is `@mlflow.trace(span_type=SpanType.TOOL)`. This tells MLflow to record each tool invocation as a `TOOL` span in the trace — capturing inputs, outputs, and timing. Without this, the tool calls would be invisible in the trace.

```python
@mlflow.trace(span_type=SpanType.TOOL)
def my_tool(arg: str) -> dict:
    ...
```

In [ ]:
#add tracing to tool

# Span Type Table

Span Types

MLflow defines standard span types:

`CHAIN`: A sequence of operations \
`LLM`: Language model call \
`RETRIEVER`: Document retrieval \
`EMBEDDING`: Text embedding \
`TOOL`: Tool/function execution \
`AGENT`: Agent reasoning \
`PARSER`: Output parsing or generic intermediate parsing of an outcome

Hierarchy Example

```
TRACE (root)
│
└─ SPAN: Agent Executor (AGENT)
   │
   ├─ SPAN: Planning Step (LLM)
   │  └─ attributes: {model: gpt-5, tokens: 50}
   │
   ├─ SPAN: Tool Execution (TOOL)
   │  └─ attributes: {tool: search, query: "..."}
   │
   └─ SPAN: Final Response (LLM)
      └─ attributes: {model: gpt-5, tokens: 150}
```

---
## 4 — Testing the Tool-Calling Agent

Let's test the same question that tripped up the no-tool agent, plus a few more.

### Check the traces

Open the MLflow UI. For the version questions, you should see:

```
TRACE (root)
│
└─ SPAN: Agent Executor (AGENT)
   │
   ├─ SPAN: ChatGoogleGenerativeAI (LLM)
   │  └─ LLM decides to call get_mlflow_version_pypi
   │
   ├─ SPAN: get_mlflow_version_pypi (TOOL)
   │  └─ tool execution returns "3.11.1"
   │
   └─ SPAN: ChatGoogleGenerativeAI (LLM)
      └─ LLM formats the result for the user
```

For the cooking questions, there should be no `TOOL` span — just a single LLM call.

---
## 6 — Building the Evaluation Dataset

Now we need eval cases that test tool-calling behavior. Each case specifies:
- `inputs` — the user's question
- `expectations.expected_tool_calls` — which tools should be called and with what arguments
- `expectations.expected_facts` (optional) — facts that should appear in the final answer

We need cases for:
1. **Should call the tool** — conversion questions
2. **Should NOT call the tool** — general cooking questions
3. **Should call the tool with correct ingredient** — density-dependent conversions

In [ ]:
eval_dataset = [
    # ── Should call get_mlflow_version_pypi ────────────────────────────────
    {
        "inputs": {"question": "What is the current version of MLflow?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "get_mlflow_version_pypi", "arguments": {}},
            ],
            "expected_facts": ["3.11.1"],
        },
    },
    {
        "inputs": {"question": "Which MLflow version is the latest release?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "get_mlflow_version_pypi", "arguments": {}},
            ],
            "expected_facts": ["3.11.1"],
        },
    },
    {
        "inputs": {"question": "Is MLflow 3.0 out yet?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "get_mlflow_version_pypi", "arguments": {}},
            ],
            "expected_facts": ["yes"],
        },
    },
    {
        "inputs": {"question": "What version of MLflow should I install for the genai features?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "get_mlflow_version_pypi", "arguments": {}},
            ],
            "expected_facts": ["3"],
        },
    },
    # ── Should NOT call get_mlflow_version_pypi ────────────────────────────
    {
        "inputs": {"question": "What's the best way to chop an onion?"},
        "expectations": {
            "expected_tool_calls": [],
        },
    },
    {
        "inputs": {"question": "How do I make scrambled eggs?"},
        "expectations": {
            "expected_tool_calls": [],
            "expected_facts": ["butter", "low heat"],
        },
    },
    {
        "inputs": {"question": "What temperature should I bake a chicken breast at?"},
        "expectations": {
            "expected_tool_calls": [],
            "expected_facts": ["375", "165"],
        },
    },
    # ── Edge case: version question embedded in a cooking context ──────────
    {
        "inputs": {"question": "I'm setting up MLflow to track my recipes app. What's the latest version I should use?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "get_mlflow_version_pypi", "arguments": {}},
            ],
            "expected_facts": ["3.11.1"],
        },
    },
]

print(f"Eval dataset: {len(eval_dataset)} examples")
print(f"  - should call tool: {sum(1 for e in eval_dataset if e['expectations'].get('expected_tool_calls'))}")
print(f"  - should NOT call tool: {sum(1 for e in eval_dataset if e['expectations'].get('expected_tool_calls') == [])}")

# Add examples to full dataset

---
## 7 — Adding Tool Call Scorers to the Pipeline

We add two new scorers to our existing pipeline:

| Scorer | What it checks | How it works |
|---|---|---|
| `ToolCallCorrectness()` | Did the agent call the right tool with the right arguments? | Compares actual tool calls in the trace against `expected_tool_calls` |
| `ToolCallEfficiency()` | Did the agent make redundant calls? | Checks for duplicate or unnecessary tool invocations |

These scorers read the **trace**, not just the text output. They look for spans with `span_type=TOOL` — which the Agents SDK creates automatically via autologging.

In [ ]:
from mlflow.genai.scorers import (
    Correctness,
    RelevanceToQuery,
    ToolCallCorrectness,
    ToolCallEfficiency,
)

results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=[
        # ── Tool quality ──────────────────────────────────────────────
        ToolCallCorrectness(),   # fuzzy match against expected_tool_calls
        ToolCallEfficiency(),    # no ground truth needed
        # ── Output quality (from Stage 1) ─────────────────────────────
        Correctness(),
        RelevanceToQuery(),
    ],
)

print("=== Evaluation with tool call scorers ===")
print(results.metrics)

In [ ]:
results.tables["eval_results_table"]

---
## 8 — Interpreting the Results

Look at the per-row breakdown above and check:

1. **Conversion questions:** Did `ToolCallCorrectness` pass? If not, inspect the trace — did the agent call `convert_units` or try to answer from memory?

2. **Non-conversion questions:** Did `ToolCallCorrectness` pass with `expected_tool_calls: []`? If the agent unnecessarily called the tool for "How do I chop an onion?", that's a tool-selection problem.

3. **Ingredient specificity:** For volume-to-weight conversions, did the agent pass the correct `ingredient` argument? Passing `"flour"` when the user said `"brown sugar"` would give the wrong answer.

4. **Efficiency:** Did `ToolCallEfficiency` flag any redundant calls? A common failure mode is calling `convert_units` twice with the same arguments.

### What to do with failures

| Failure | Root cause | Fix |
|---|---|---|
| Agent doesn't call the tool | System prompt doesn't emphasize tool use enough | Update the `instructions` |
| Agent calls tool for non-conversion questions | Tool description is too broad | Narrow the `@function_tool` docstring |
| Wrong ingredient argument | LLM extracted the wrong ingredient from the question | Improve the tool's `Args` description |
| Redundant calls | LLM called the tool twice | Usually a model-level issue, may need prompt tuning |

---
## 9 — Before and After: No Tool vs Tool

Let's run the conversion subset against the no-tool agent to quantify the improvement.

In [ ]:
def predict_fn_no_tools(inputs: dict) -> str:
    """Adapter for the no-tool agent."""
    result = Runner.run_sync(agent_no_tools, inputs["question"])
    return result.final_output


# Only the conversion questions (where we expect the tool to help)
conversion_subset = [e for e in eval_dataset if e["expectations"].get("expected_tool_calls")]

results_no_tool = mlflow.genai.evaluate(
    data=conversion_subset,
    predict_fn=predict_fn_no_tools,
    scorers=[Correctness()],
)

results_with_tool = mlflow.genai.evaluate(
    data=conversion_subset,
    predict_fn=predict_fn,
    scorers=[Correctness()],
)

print("=== Conversion questions: No tool vs With tool ===")
print(f"No tool   — Correctness: {results_no_tool.metrics}")
print(f"With tool — Correctness: {results_with_tool.metrics}")

---
## Summary: What We Built

| Concept | How we used it |
|---|---|
| Product conversation | Sam's feedback → unit conversion tool requirement |
| `@function_tool` | Defined `convert_units` with density table for ingredient-specific conversions |
| `Agent(tools=[...])` | Added the tool to the existing agent — one line |
| Updated system prompt | Told the agent when and why to use the tool |
| `expected_tool_calls` | Ground truth for which tool should be called and with what arguments |
| `ToolCallCorrectness()` | Evaluates tool selection and argument accuracy |
| `ToolCallEfficiency()` | Catches redundant or unnecessary tool calls |
| Before/after comparison | Quantified the improvement from adding the tool |

### The pattern

```
Product feedback ("users are getting wrong conversions")
  → Identify the root cause (LLM guessing at density-dependent math)
  → Build a deterministic tool (conversion table with known densities)
  → Add tool to agent + update system prompt
  → Add eval cases with expected_tool_calls
  → Add ToolCallCorrectness + ToolCallEfficiency to scorer suite
  → Run eval to confirm improvement
  → Compare before/after
```

---

**Next →** We'll add more tools to the agent and explore what happens when the agent has to choose between multiple tools — tool selection becomes a harder problem.